In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")

from src.db import get_engine

engine = get_engine()

PROJECT_ROOT


PosixPath('/home/wyl/projects/soft-soil-ml')

In [2]:
machine_code = "RIG-001"
project_code = "NBU-WEST-DEMO"
borehole_code = "BH-001"

rig_df = pd.read_sql(
    """
    SELECT
        timestamp,
        machine_code,
        project_code,
        borehole_code,
        drilling_depth,
        current_value,
        torque,
        penetration_speed,
        grouting_pressure,
        rotation_speed_rpm,
        mud_density_g_cm3,
        pump_flow_l_min,
        verticality_deg,
        latitude,
        longitude
    FROM machine_realtime_data
    WHERE machine_code = %(machine_code)s
      AND project_code = %(project_code)s
      AND borehole_code = %(borehole_code)s
    ORDER BY timestamp
    """,
    engine,
    params={
        "machine_code": machine_code,
        "project_code": project_code,
        "borehole_code": borehole_code,
    },
)

rig_df.head()


,timestamp,machine_code,project_code,borehole_code,drilling_depth,current_value,torque,penetration_speed,grouting_pressure,rotation_speed_rpm,mud_density_g_cm3,pump_flow_l_min,verticality_deg,latitude,longitude
0,2026-05-12 08:00:00,RIG-001,NBU-WEST-DEMO,BH-001,0.0000,146.005,56.568,0.08322,1.0979,28.648,1.1879,124.211,0.2482,29.886200,121.556799
1,2026-05-12 08:00:01,RIG-001,NBU-WEST-DEMO,BH-001,0.0013,126.508,56.424,0.08838,1.2192,28.716,1.1827,111.208,0.2201,29.886202,121.556802
2,2026-05-12 08:00:02,RIG-001,NBU-WEST-DEMO,BH-001,0.0027,126.033,48.368,0.08581,1.4308,29.285,1.1722,117.121,0.2738,29.886204,121.556799
3,2026-05-12 08:00:03,RIG-001,NBU-WEST-DEMO,BH-001,0.0040,148.137,47.666,0.09337,1.4932,28.540,1.1768,113.236,0.2619,29.886199,121.556800
4,2026-05-12 08:00:04,RIG-001,NBU-WEST-DEMO,BH-001,0.0053,127.671,57.934,0.08291,1.2541,28.199,1.1820,120.405,0.2347,29.886199,121.556800


In [3]:
feature_cols = [
    "current_value",
    "torque",
    "penetration_speed",
    "grouting_pressure",
    "rotation_speed_rpm",
    "mud_density_g_cm3",
    "pump_flow_l_min",
    "verticality_deg",
]

work_df = df.copy()
work_df["timestamp"] = pd.to_datetime(work_df["timestamp"])

work_df = work_df.dropna(subset=["drilling_depth"]).copy()

for col in feature_cols:
    work_df[col] = pd.to_numeric(work_df[col], errors="coerce")
    work_df[col] = work_df[col].interpolate().ffill().bfill()

work_df = work_df.sort_values("drilling_depth").reset_index(drop=True)

print("数据规模:", work_df.shape)
print("深度范围:", work_df["drilling_depth"].min(), work_df["drilling_depth"].max())

work_df.head()


NameError: name 'df' is not defined

In [ ]:
dri_cols = [
    "current_value",
    "torque",
    "grouting_pressure",
    "penetration_speed",
]

dri_values = work_df[dri_cols]

dri_z = (
    dri_values - dri_values.mean()
) / dri_values.std(ddof=0).replace(0, 1)

work_df["DRI"] = (
    0.30 * dri_z["current_value"]
    + 0.30 * dri_z["torque"]
    + 0.25 * dri_z["grouting_pressure"]
    - 0.15 * dri_z["penetration_speed"]
)

work_df[
    [
        "drilling_depth",
        "current_value",
        "torque",
        "penetration_speed",
        "grouting_pressure",
        "DRI",
    ]
].head()


In [ ]:
depth_bin_m = 0.05

depth_source = work_df.copy()

# 过滤明显停钻/接杆段
depth_source = depth_source[
    depth_source["penetration_speed"].fillna(0) > 0.001
].copy()

depth_source["depth_bin"] = (
    (depth_source["drilling_depth"] / depth_bin_m).round() * depth_bin_m
)

depth_df = (
    depth_source
    .groupby("depth_bin", as_index=False)
    .agg({
        "current_value": "median",
        "torque": "median",
        "penetration_speed": "median",
        "grouting_pressure": "median",
        "DRI": "median",
    })
    .rename(columns={"depth_bin": "drilling_depth"})
    .sort_values("drilling_depth")
    .reset_index(drop=True)
)

print("重采样后数据规模:", depth_df.shape)
depth_df.head()


In [ ]:
signal_cols = [
    "current_value",
    "torque",
    "penetration_speed",
    "grouting_pressure",
    "rotation_speed_rpm",
    "mud_density_g_cm3",
    "pump_flow_l_min",
    "verticality_deg",
    "DRI",
]


X = depth_df[signal_cols].copy()

median = X.median()
iqr = X.quantile(0.75) - X.quantile(0.25)
iqr = iqr.replace(0, 1)

Z = (X - median) / iqr

Z_smooth = Z.rolling(
    window=5,
    center=True,
    min_periods=1
).median()

Z_smooth.head()


In [ ]:
left_right_window_m = 0.5
window_bins = max(3, int(left_right_window_m / depth_bin_m))

Z_values = Z_smooth.values
scores = []

for i in range(len(depth_df)):
    left_start = max(0, i - window_bins)
    left_end = i
    right_start = i + 1
    right_end = min(len(depth_df), i + 1 + window_bins)

    if (
        left_end - left_start < window_bins // 2
        or right_end - right_start < window_bins // 2
    ):
        scores.append(0.0)
        continue

    left_mean = Z_values[left_start:left_end].mean(axis=0)
    right_mean = Z_values[right_start:right_end].mean(axis=0)

    score = np.linalg.norm(right_mean - left_mean)
    scores.append(score)

depth_df["change_score"] = scores

depth_df["change_score_smooth"] = (
    depth_df["change_score"]
    .rolling(window=5, center=True, min_periods=1)
    .mean()
)

depth_df[["drilling_depth", "change_score_smooth"]].head()


In [ ]:
score_quantile = 0.90
min_gap_m = 1.2

threshold = depth_df["change_score_smooth"].quantile(score_quantile)

candidate_df = depth_df[
    depth_df["change_score_smooth"] >= threshold
].copy()

boundary_rows = []

for _, row in candidate_df.sort_values("drilling_depth").iterrows():
    depth = row["drilling_depth"]

    if not boundary_rows:
        boundary_rows.append(row)
        continue

    last_depth = boundary_rows[-1]["drilling_depth"]

    if depth - last_depth < min_gap_m:
        if row["change_score_smooth"] > boundary_rows[-1]["change_score_smooth"]:
            boundary_rows[-1] = row
    else:
        boundary_rows.append(row)

detected_boundaries_df = pd.DataFrame(boundary_rows)

if not detected_boundaries_df.empty:
    detected_boundaries_df = (
        detected_boundaries_df[
            ["drilling_depth", "change_score_smooth"]
        ]
        .rename(columns={"drilling_depth": "boundary_depth"})
        .reset_index(drop=True)
    )

print("阈值:", threshold)
detected_boundaries_df


In [ ]:
boundaries = (
    detected_boundaries_df["boundary_depth"]
    .dropna()
    .sort_values()
    .tolist()
)

min_depth = depth_df["drilling_depth"].min()
max_depth = depth_df["drilling_depth"].max()

layer_edges = [min_depth] + boundaries + [max_depth]

layer_segments = []

for i in range(len(layer_edges) - 1):
    start_depth = layer_edges[i]
    end_depth = layer_edges[i + 1]

    if i == 0:
        mask = (
            (depth_df["drilling_depth"] >= start_depth) &
            (depth_df["drilling_depth"] <= end_depth)
        )
    else:
        mask = (
            (depth_df["drilling_depth"] > start_depth) &
            (depth_df["drilling_depth"] <= end_depth)
        )

    seg = depth_df[mask]

    layer_segments.append({
        "detected_layer_index": i + 1,
        "start_depth": start_depth,
        "end_depth": end_depth,
        "thickness": end_depth - start_depth,
        "sample_count": len(seg),
        "current_mean": seg["current_value"].mean(),
        "torque_mean": seg["torque"].mean(),
        "penetration_speed_mean": seg["penetration_speed"].mean(),
        "grouting_pressure_mean": seg["grouting_pressure"].mean(),
        "DRI_mean": seg["DRI"].mean(),
        "change_score_mean": seg["change_score_smooth"].mean(),
    })

detected_layers_df = pd.DataFrame(layer_segments)

detected_layers_df


In [ ]:
def assign_detected_layer(depth, edges):
    for i in range(len(edges) - 1):
        start_depth = edges[i]
        end_depth = edges[i + 1]

        if i == 0:
            if start_depth <= depth <= end_depth:
                return i + 1
        else:
            if start_depth < depth <= end_depth:
                return i + 1

    return np.nan


depth_df["detected_layer_index"] = depth_df["drilling_depth"].apply(
    lambda d: assign_detected_layer(d, layer_edges)
)

depth_df[["drilling_depth", "detected_layer_index"]].head()


In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(
    depth_df["drilling_depth"],
    depth_df["change_score_smooth"],
    label="change score"
)

for boundary in boundaries:
    plt.axvline(
        boundary,
        color="red",
        linestyle="--",
        alpha=0.7
    )

plt.xlabel("drilling_depth (m)")
plt.ylabel("change score")
plt.title("Detected Soil Change Boundaries")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


In [ ]:
plot_cols = [
    "current_value",
    "torque",
    "grouting_pressure",
    "DRI",
]

plt.figure(figsize=(14, 6))

for col in plot_cols:
    y = depth_df[col]
    y_norm = (y - y.min()) / (y.max() - y.min() + 1e-9)

    plt.plot(
        depth_df["drilling_depth"],
        y_norm,
        label=col
    )

for boundary in boundaries:
    plt.axvline(
        boundary,
        color="red",
        linestyle="--",
        alpha=0.7
    )

for _, row in detected_layers_df.iterrows():
    mid = (row["start_depth"] + row["end_depth"]) / 2

    plt.text(
        mid,
        1.03,
        f"L{int(row['detected_layer_index'])}",
        ha="center",
        va="bottom"
    )

plt.xlabel("drilling_depth (m)")
plt.ylabel("normalized signal")
plt.title("Detected Layer Segments from Rig Data")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


In [ ]:
detected_layers_df[
    [
        "detected_layer_index",
        "start_depth",
        "end_depth",
        "thickness",
        "sample_count",
        "current_mean",
        "torque_mean",
        "penetration_speed_mean",
        "grouting_pressure_mean",
        "DRI_mean",
    ]
]
